# Chapter 6 — Prompt Engineering

**Book:** Hands-On Large Language Models  
**Chapter:** 6 — Methods for improving model output through prompt design

---

## What You Will Learn

Prompt engineering is the art of communicating with a language model through carefully designed text inputs. This chapter covers:

- The **anatomy of a prompt** and the seven components that shape model behaviour
- **In-context learning**: teaching models new tasks using only examples in the prompt
- **Chain prompting**: decomposing complex tasks into sequential steps
- **Chain-of-Thought (CoT)**: eliciting step-by-step reasoning from a model
- **Tree-of-Thought (ToT)**: having multiple reasoning paths compete and converge
- **Output verification**: controlling the format and structure of model responses
- **Constrained sampling**: using grammar-based sampling to guarantee valid JSON

---

## Parts Overview

| Part | Topic |
|------|-------|
| 1 | Loading the model + generation parameters |
| 2 | Prompt anatomy — the 7 components |
| 3 | In-context learning + chain prompting |
| 4 | Reasoning: CoT and Tree-of-Thought |
| 5 | Output verification + constrained sampling |

---

# Part 1 — Loading the Model and Generation Parameters

We use `microsoft/Phi-3-mini-4k-instruct` throughout this chapter. It is a small but capable instruction-tuned model that supports a chat template, making it ideal for demonstrating prompt engineering techniques.

The `pipeline` abstraction from HuggingFace wraps the tokenizer, model forward pass, and decoding into a single callable. We set `return_full_text=False` so we only see the model's continuation, not the prompt repeated back.

In [ ]:
# %%capture
# !pip install transformers>=4.40.1 accelerate>=0.27.2

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False,
)

print("Model loaded successfully")
print(f"Model device: {model.device}")

### 1.1 Basic Generation

A message is a list of dicts with `role` ("user" or "assistant") and `content`. The model generates a continuation given the conversation history.

In [ ]:
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]

output = pipe(messages)
print(output[0]["generated_text"])

### 1.2 The Chat Template

Under the hood, the model does not receive a Python list — it receives a single string formatted according to its **chat template**. Each model family has its own template. For Phi-3:

```
<s><|user|>
your message<|end|>
<|assistant|>
```

You can inspect what the model actually sees by calling `apply_chat_template`.

In [ ]:
# Inspect the raw string that gets sent to the model
prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False)
print(prompt)

### 1.3 Theory Exercise — Generation Parameters

Two key parameters control how the model samples the next token:

**Temperature** (`temperature`): scales the logits before softmax. High temperature → flatter distribution → more random output. Low temperature → sharper distribution → more predictable output. `do_sample=False` (greedy) is equivalent to temperature → 0.

**Top-p / Nucleus Sampling** (`top_p`): at each step, sort tokens by probability descending and keep only the smallest set whose cumulative probability exceeds `top_p`. Sample from that set only. This prevents the model from sampling extremely low-probability tokens.

Run both cells below and observe how the outputs differ from greedy decoding.

In [ ]:
# High temperature — more creative/random
output = pipe(messages, do_sample=True, temperature=1)
print("Temperature=1 output:")
print(output[0]["generated_text"])

In [ ]:
# Nucleus sampling with top_p
output = pipe(messages, do_sample=True, top_p=1)
print("top_p=1 output:")
print(output[0]["generated_text"])

**Reflection:** Why does `top_p=1` still produce different outputs from greedy decoding even though it includes all tokens in the nucleus?

---

# Part 2 — The Anatomy of a Prompt

Most production prompts are not single sentences. They are structured documents composed of up to **seven components**:

| Component | Purpose | Example |
|-----------|---------|--------|
| **Persona** | Sets the model's role and expertise | "You are an expert in Large Language Models..." |
| **Instruction** | States the primary task | "Summarize the key findings of the paper." |
| **Context** | Provides background that shapes the answer | "Your summary should help researchers quickly grasp..." |
| **Format** | Specifies structure of the output | "Create a bullet-point summary followed by a paragraph." |
| **Audience** | Describes who will read the output | "For busy researchers who need the latest trends." |
| **Tone** | Sets the register and style | "Professional and clear." |
| **Data** | The actual content to process | "Text to summarize: [text]" |

Not all seven are needed for every task — but each one you add gives the model more signal to produce exactly what you want.

### 2.1 Building a Complex Prompt

Each component is a string. We concatenate them into a single `query` and pass it as the user message.

In [ ]:
# A text excerpt to summarize (from the Illustrated Transformer blog)
text = """In the previous post, we looked at Attention – a ubiquitous method in
modern deep learning models. Attention is a concept that helped improve the
performance of neural machine translation applications. In this post, we will
look at The Transformer – a model that uses attention to speed up training.
The Transformers outperforms the Google Neural Machine Translation model in
specific tasks. The biggest benefit, however, comes from how The Transformer
lends itself to parallelization."""

# The 7 prompt components
persona      = "You are an expert in Large Language models. You excel at breaking down complex papers into digestible summaries.\n"
instruction  = "Summarize the key findings of the paper provided.\n"
context      = "Your summary should extract the most crucial points that help researchers quickly understand the most vital information of the paper.\n"
data_format  = "Create a bullet-point summary that outlines the method. Follow this up with a concise paragraph that encapsulates the main results.\n"
audience     = "The summary is designed for busy researchers that quickly need to grasp the newest trends in Large Language Models.\n"
tone         = "The tone should be professional and clear.\n"
data         = f"Text to summarize: {text}"

query = persona + instruction + context + data_format + audience + tone + data

print(f"Prompt length: {len(query)} characters")
print()
print(query)

In [ ]:
messages = [{"role": "user", "content": query}]

# Inspect the tokenized prompt before sending
print(tokenizer.apply_chat_template(messages, tokenize=False))

In [ ]:
outputs = pipe(messages)
print(outputs[0]["generated_text"])

### 2.2 Theory Exercise — Ablation Study

An **ablation study** removes one component at a time to measure its contribution. Below, define three variations of the prompt:

1. **Full prompt** — all 7 components (already done above)
2. **No persona** — remove the persona component, keep the rest
3. **Instruction only** — just instruction + data (minimal prompt)

Run each variant and compare the outputs. Pay attention to: tone, structure, detail level, and whether the output matches the requested format.

In [ ]:
# Variant 2: Remove the persona component
# YOUR CODE HERE
query_no_persona = ""

messages_no_persona = [{"role": "user", "content": query_no_persona}]
output_no_persona = pipe(messages_no_persona)
print("Without persona:")
print(output_no_persona[0]["generated_text"])

In [ ]:
# Variant 3: Minimal prompt — instruction + data only
# YOUR CODE HERE
query_minimal = ""

messages_minimal = [{"role": "user", "content": query_minimal}]
output_minimal = pipe(messages_minimal)
print("Minimal prompt (instruction + data only):")
print(output_minimal[0]["generated_text"])

**Reflection:** Which components had the biggest impact on output quality and format? Which seemed to make little difference for this specific task?

---

# Part 3 — In-Context Learning and Chain Prompting

## 3.1 In-Context Learning: Providing Examples

**In-context learning** is the ability of large language models to learn a new task purely from examples embedded in the prompt — no gradient updates, no fine-tuning.

The model sees a pattern in the conversation history and continues it:
- **Zero-shot**: no examples, just a task description
- **One-shot**: one example before the actual query
- **Few-shot**: multiple examples

The example below uses **made-up words** that the model cannot have seen in pre-training. This proves the model is genuinely learning the pattern from context, not retrieving memorised knowledge.

### Theory Exercise — Token Count Check

Before running the prompt, count how many tokens the one-shot prompt uses. This matters because:
- Every token costs compute during inference
- Models have a finite context window
- Longer prompts can push the model's response out of the context window

Implement the token count check below.

In [ ]:
one_shot_prompt = [
    {
        "role": "user",
        "content": "A 'Gigamuru' is a type of Japanese musical instrument. An example of a sentence that uses the word Gigamuru is:"
    },
    {
        "role": "assistant",
        "content": "I have a Gigamuru that my uncle gave me as a gift. I love to play it at home."
    },
    {
        "role": "user",
        "content": "To 'screeg' something is to swing a sword at it. An example of a sentence that uses the word screeg is:"
    }
]

# Count how many tokens this prompt uses
# Hint: use tokenizer.apply_chat_template(..., tokenize=True) and check its length
# YOUR CODE HERE
token_count = 0

print(f"One-shot prompt token count: {token_count}")

# Show the raw formatted prompt
print(tokenizer.apply_chat_template(one_shot_prompt, tokenize=False))

In [ ]:
outputs = pipe(one_shot_prompt)
print(outputs[0]["generated_text"])

**Observation:** The model correctly uses "screeg" as a verb (past tense) applied to a physical object — behaviour it inferred entirely from one example. Notice how the model mirrors the sentence structure from the Gigamuru example.

## 3.2 Chain Prompting: Breaking Up the Problem

**Chain prompting** splits a complex task into a sequence of simpler prompts where the **output of one step becomes the input of the next**.

This approach works because:
- Each step is a focused, tractable sub-task
- Intermediate results can be inspected and validated
- Different models or parameters can be used at each stage

We demonstrate a two-step chain: generate a product name and slogan → use that output to generate a sales pitch.

```
Step 1: [Task Description] ──► Model ──► product_description
                                               │
Step 2:                    f(product_description) ──► Model ──► sales_pitch
```

In [ ]:
# Step 1: Generate a product name and slogan
product_prompt = [
    {"role": "user", "content": "Create a name and slogan for a chatbot that leverages LLMs."}
]
outputs = pipe(product_prompt)
product_description = outputs[0]["generated_text"]

print("Step 1 output (product name + slogan):")
print(product_description)

In [ ]:
# Step 2: Use Step 1 output to generate a sales pitch
# The key: product_description from Step 1 is injected into Step 2's prompt
sales_prompt = [
    {"role": "user", "content": f"Generate a very short sales pitch for the following product: '{product_description}'"}
]
outputs = pipe(sales_prompt)
sales_pitch = outputs[0]["generated_text"]

print("Step 2 output (sales pitch):")
print(sales_pitch)

### Theory Exercise — Design a 3-Step Chain

Extend the chain by adding a **third step** that uses the `sales_pitch` to generate a **10-word Twitter/X post** promoting the product.

Requirements:
- The post must reference the product name from `product_description`
- It must be under 280 characters
- The prompt should include those constraints explicitly

In [ ]:
# Step 3: Generate a short social media post from the sales pitch
# YOUR CODE HERE
tweet_prompt = []

# outputs = pipe(tweet_prompt)
# tweet = outputs[0]["generated_text"]
# print("Step 3 output (tweet):")
# print(tweet)

---

# Part 4 — Reasoning with Generative Models

## 4.1 Chain-of-Thought: Think Before Answering

**Chain-of-Thought (CoT) prompting** elicits step-by-step reasoning from the model by providing an example where the correct answer is arrived at through explicit intermediate steps.

The key insight: language models generate text left-to-right. If you force the model to write out its reasoning *before* writing the final answer, that reasoning becomes part of the context for the answer token. This dramatically improves performance on multi-step arithmetic, logical reasoning, and symbolic manipulation.

Without CoT, the model jumps straight to an answer. With CoT, each reasoning step serves as a "scratchpad" that constrains the next step.

```
Without CoT:  [Problem] → "11"  (direct, potentially wrong)
With CoT:     [Problem] → "Roger started with 5 balls. 2 cans × 3 = 6 balls. 5 + 6 = 11. The answer is 11."
```

In [ ]:
# Few-shot Chain-of-Thought: one worked example teaches the reasoning pattern
cot_prompt = [
    {
        "role": "user",
        "content": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"
    },
    {
        "role": "assistant",
        "content": "Roger started with 5 balls. 2 cans of 3 tennis balls each is 6 tennis balls. 5 + 6 = 11. The answer is 11."
    },
    {
        "role": "user",
        "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?"
    }
]

outputs = pipe(cot_prompt)
print(outputs[0]["generated_text"])

## 4.2 Zero-Shot Chain-of-Thought

Instead of providing a worked example, we can trigger CoT behaviour with a **single magic phrase**: `"Let's think step-by-step."`

This works because models trained on large corpora have seen countless step-by-step explanations. Appending this phrase shifts the model's generation distribution toward structured reasoning without requiring an explicit example.

This is a **zero-shot** technique — no examples in the prompt.

In [ ]:
zeroshot_cot_prompt = [
    {
        "role": "user",
        "content": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have? Let's think step-by-step."
    }
]

outputs = pipe(zeroshot_cot_prompt)
print(outputs[0]["generated_text"])

### Theory Exercise — Compare Standard vs CoT

Run the same cafeteria problem **without** the "Let's think step-by-step" trigger. Compare the two outputs.

Then try a **harder multi-step problem** of your own design with both approaches. The gap between standard and CoT prompting typically widens as problem complexity increases.

In [ ]:
# Standard prompt — no CoT trigger
# YOUR CODE HERE
standard_prompt = []

# outputs = pipe(standard_prompt)
# print("Standard (no CoT):")
# print(outputs[0]["generated_text"])

In [ ]:
# Design a harder multi-step problem and compare standard vs CoT
# Example: a problem involving multiple operations, units, or conditions
# YOUR CODE HERE
hard_problem = ""  # define your problem here

# hard_standard = [{"role": "user", "content": hard_problem}]
# hard_cot = [{"role": "user", "content": hard_problem + " Let's think step-by-step."}]

## 4.3 Tree-of-Thought: Exploring Multiple Reasoning Paths

**Tree-of-Thought (ToT)** extends CoT by having **multiple reasoning agents** work simultaneously, sharing progress at each step. If one agent reaches an inconsistency, it drops out. The surviving agents converge on a common answer.

This simulates the *peer review* process: multiple perspectives cross-check each other, reducing the chance that a single flawed chain of reasoning reaches an incorrect conclusion.

```
Expert A: Step 1 → shares → Step 2 → shares → ...
Expert B: Step 1 → shares → exits (found error) ✗
Expert C: Step 1 → shares → Step 2 → shares → ... → final answer
```

In practice, this is implemented as a single prompt that instructs the model to roleplay as three different experts.

In [ ]:
zeroshot_tot_prompt = [
    {
        "role": "user",
        "content": (
            "Imagine three different experts are answering this question. "
            "All experts will write down 1 step of their thinking, then share it with the group. "
            "Then all experts will go on to the next step, etc. "
            "If any expert realises they're wrong at any point then they leave. "
            "The question is: 'The cafeteria had 23 apples. If they used 20 to make lunch and "
            "bought 6 more, how many apples do they have?' Make sure to discuss the results."
        )
    }
]

outputs = pipe(zeroshot_tot_prompt)
print(outputs[0]["generated_text"])

### Theory Exercise — ToT vs CoT Analysis

Compare Tree-of-Thought and zero-shot CoT on the same cafeteria problem:

1. Did both arrive at the same final answer?
2. Which output is more verbose?
3. For what types of problems would ToT provide the most benefit over CoT? (Think: problems with many valid solution paths, problems where it is easy to make an error in one step, etc.)

Write your analysis below.

In [ ]:
# YOUR ANALYSIS (as a print statement or markdown cell above)
analysis = {
    "same_answer": None,      # True or False
    "more_verbose": None,     # "CoT" or "ToT"
    "tot_best_for": None,     # string describing the problem types
}
print(analysis)

---

# Part 5 — Output Verification and Constrained Sampling

## 5.1 Format Control via Examples

When you need structured output (JSON, YAML, tables), you can guide the model with an **example of the expected format** embedded in the prompt.

- **Zero-shot**: ask for JSON, get an inconsistent structure (model decides field names)
- **One-shot**: show an example schema, model mirrors the exact structure

The one-shot approach is reliable for simple schemas. For complex schemas with strict validation, constrained sampling is necessary.

In [ ]:
# Zero-shot: ask for JSON with no example — model invents its own schema
zeroshot_prompt = [
    {"role": "user", "content": "Create a character profile for an RPG game in JSON format."}
]

outputs = pipe(zeroshot_prompt)
print("Zero-shot JSON output:")
print(outputs[0]["generated_text"])

In [ ]:
# One-shot: provide the exact schema as a template in the prompt
one_shot_template = """Create a short character profile for an RPG game. Make sure to only use this format:

{
  "description": "A SHORT DESCRIPTION",
  "name": "THE CHARACTER'S NAME",
  "armor": "ONE PIECE OF ARMOR",
  "weapon": "ONE OR MORE WEAPONS"
}
"""
one_shot_prompt = [{"role": "user", "content": one_shot_template}]

outputs = pipe(one_shot_prompt)
print("One-shot JSON output:")
print(outputs[0]["generated_text"])

### Theory Exercise — Parse and Validate

The one-shot output should be valid JSON, but is it? Use Python's `json` module to parse the output. If parsing fails, explain why and fix the prompt to make it more reliable.

In [ ]:
import json

raw_output = outputs[0]["generated_text"].strip()

# Attempt to parse the one-shot output as JSON
# YOUR CODE HERE — wrap in try/except and print whether it parsed successfully
try:
    parsed = None  # replace with json.loads(...)
    print("Parsed successfully:")
    print(json.dumps(parsed, indent=2))
except json.JSONDecodeError as e:
    print(f"Parse failed: {e}")
    print("Raw output was:")
    print(raw_output)

## 5.2 Grammar-Based Constrained Sampling

Prompt-based format control works most of the time, but not always. For applications where invalid JSON would crash downstream code, we need **constrained sampling** — a technique that modifies the sampling process to only allow tokens that are consistent with a formal grammar (e.g., JSON schema).

This is different from prompting: instead of *asking* the model to produce JSON, we *enforce* it by masking out any token at each step that would make the output invalid.

We use `llama-cpp-python` with a GGUF-format model for this. The library implements a finite-state machine over the JSON grammar and adjusts the logit mask at each decoding step.

**Note:** We first free GPU memory used by the HuggingFace model before loading the llama-cpp model.

In [ ]:
import gc
import torch

# Free GPU memory from the HuggingFace model
del model, tokenizer, pipe
gc.collect()
torch.cuda.empty_cache()

print("GPU memory freed")

In [ ]:
# %%capture
# !CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install llama-cpp-python

In [ ]:
from llama_cpp.llama import Llama

# Load Phi-3 in GGUF format via llama-cpp
llm = Llama.from_pretrained(
    repo_id="microsoft/Phi-3-mini-4k-instruct-gguf",
    filename="*fp16.gguf",
    n_gpu_layers=-1,
    n_ctx=2048,
    verbose=False,
)

print("llama-cpp model loaded")

In [ ]:
# Generate with constrained sampling — response_format enforces valid JSON
# YOUR CODE HERE
# Use llm.create_chat_completion() with:
#   messages=[{"role": "user", "content": "Create a warrior for an RPG in JSON format."}]
#   response_format={"type": "json_object"}
#   temperature=0
# Extract the content from the response

output = None  # YOUR CODE HERE

In [ ]:
# Parse and pretty-print the constrained output
# YOUR CODE HERE
# json_output = json.dumps(json.loads(output), indent=4)
# print(json_output)

### Theory Exercise — Compare Approaches

You have now seen three ways to get JSON output from a model:

1. **Zero-shot**: ask for JSON in a natural language prompt
2. **One-shot**: provide an example schema in the prompt
3. **Constrained sampling**: enforce JSON grammar at the token level

Complete the table below:

| Approach | Guarantees valid JSON? | Controls field names? | Compute overhead | Requires special library? |
|----------|----------------------|----------------------|-----------------|---------------------------|
| Zero-shot | | | | |
| One-shot | | | | |
| Constrained sampling | | | | |

**When would you use each approach in production?**

---

# Chapter Summary

## Key Concepts

**Prompt anatomy**: A well-engineered prompt has up to 7 components — persona, instruction, context, format, audience, tone, and data. Each adds specificity and reduces ambiguity. Adding all 7 is not always necessary; ablation studies reveal which components matter for each task.

**In-context learning**: Large models can learn new tasks from examples embedded in the prompt. Zero-shot (no examples), one-shot (one example), and few-shot (multiple examples) refer to how many demonstrations you provide. The model generalises the pattern rather than memorising the specific example.

**Chain prompting**: Breaking a complex task into sequential prompts where each output feeds the next. Enables validation at intermediate stages and allows different generation strategies per step.

**Chain-of-Thought (CoT)**: Appending step-by-step reasoning to the prompt forces the model to produce intermediate computation before the final answer. Because generation is left-to-right, correct intermediate tokens constrain subsequent tokens toward correct answers. The phrase "Let's think step-by-step" is sufficient to trigger this in many models (zero-shot CoT).

**Tree-of-Thought (ToT)**: Multiple reasoning paths explored simultaneously, with agents that drop out if they find an error. Reduces the risk of a single flawed reasoning chain producing a wrong answer. Computationally more expensive than CoT.

**Output verification**: One-shot format examples in the prompt guide the model toward a specific output structure. Constrained sampling (using llama-cpp's `response_format`) enforces structure at the token level with mathematical guarantees.

## Prompt Engineering Decision Tree

```
Is output format critical (e.g., must parse as JSON)?
├── YES → Use constrained sampling (llama-cpp + response_format)
└── NO  →
    Is the task complex / multi-step reasoning?
    ├── YES →
    │    Do you need multiple independent perspectives?
    │    ├── YES → Tree-of-Thought
    │    └── NO  → Chain-of-Thought (few-shot or zero-shot)
    └── NO  →
         Do you have labelled task examples available?
         ├── YES → Few-shot in-context learning
         └── NO  → Zero-shot with full 7-component prompt
```